# 🧩 Prerequisites

In [ ]:
# @title 📀 Mount Google Drive
from google.colab import drive
from pathlib import Path
drive.mount("/content/drive")
%cd /content/drive/MyDrive/colab/thesis/rrs
CWD = Path.cwd()

In [ ]:
# @title 📦 Package Installation
%%capture
!pip install uv
!uv pip install evaluate rouge_score bert-score
!uv pip install ctranslate2
!uv pip install transformers==4.56.2

# 📊 Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("adasatwork-thesis/rrs", data_files={
    "test" : "test.jsonl"
}, split="test")

print(dataset)

# 🧬 BioBART Model

In [ ]:
# @title BioBART Inference Using CTranslate2
import ctranslate2
from transformers import AutoTokenizer

class BioBART:
    def __init__(self, model_name):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.translator = ctranslate2.Translator(
            model_name + "-ct2",
            device="cuda",
        )

    def generate(self, texts):
        inputs = self.tokenizer(
            texts,
            max_length=512,
            truncation=True
        )
        batch_tokens = [
            self.tokenizer.convert_ids_to_tokens(ids)
            for ids in inputs["input_ids"]
        ]

        results = self.translator.translate_batch(
            batch_tokens,
            beam_size=4,
            patience=1.5,
            max_decoding_length=256,
            length_penalty=0.8,
            repetition_penalty=1.5,
            max_batch_size=64
        )

        outputs = self.tokenizer.batch_decode(
            [self.tokenizer.convert_tokens_to_ids(r.hypotheses[0]) for r in results],
            skip_special_tokens=True
        )

        return outputs

# 🔮 Qwen3 Model

In [ ]:
# @title Qwen3 Inference Using CTranslate2
import re
import ctranslate2
from transformers import AutoTokenizer

# clean_pattern = re.compile(r"(?:\n|/\\|\\/|\s+)")

system_text = """You are a Medical Radiology Hallucination Detector and Corrector."""
user_text = """\
# Instructions:
1. Compare the [RADIOLOGY FINDINGS] with its [RADIOLOGY IMPRESSION].
2. Identify the incorrect detail in the [RADIOLOGY IMPRESSION] and fix it using the [RADIOLOGY FINDINGS].
3. Output ONLY the final [CORRECTED RADIOLOGY IMPRESSION] with minimal text.

# Context:
[RADIOLOGY FINDINGS]: {}
[RADIOLOGY IMPRESSION]: {}

# Output:
[CORRECTED RADIOLOGY IMPRESSION]:"""


class Qwen3:
    def __init__(self, model_name):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.generator = ctranslate2.Generator(model_name, device="cuda")

    def generate(self, sources, biobart_gens):
        # Build the static (system) prompt tokens
        system_messages = [{"role" : "system", "content" : system_text}]
        system_prompt = self.tokenizer.apply_chat_template(
            system_messages, tokenize=False, add_generation_prompt=False
        )
        system_tokens = self.tokenizer.convert_ids_to_tokens(
            self.tokenizer.encode(system_prompt, add_special_tokens=False)
        )

        # Build per-request user prompt tokens
        batch_tokens = []
        for source, biobart_gen in zip(sources, biobart_gens):
            messages = [{
                "role" : "user",
                "content" : user_text.format(source, biobart_gen)
            }]

            prompt = self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
            input_ids = self.tokenizer.encode(prompt, add_special_tokens=False)
            batch_tokens.append(self.tokenizer.convert_ids_to_tokens(input_ids))

        # Run batch generation
        results = self.generator.generate_batch(
            batch_tokens,
            static_prompt=system_tokens,
            max_length=64,
            max_batch_size=1,
            sampling_topk=1, # Greedy decoding for the fastest possible output
            include_prompt_in_result=False
        )

        # Decode and output
        outputs = self.tokenizer.batch_decode(
            [r.sequences_ids[0] for r in results],
            skip_special_tokens=True
        )

        return outputs

# 🔴 Final Model

In [ ]:
bart = BioBART("models/finetuned_biobart")
qwen = Qwen3("models/qwen3-sft-merged-ct2")
qwnx = Qwen3("models/qwen3-simpo-merged-ct2")

In [ ]:
import pandas as pd
test_df = dataset.to_pandas()

In [ ]:
sources = test_df["findings"].to_list()
biogens = bart.generate(sources)

In [ ]:
qwngens = qwen.generate(sources, biogens)

In [ ]:
qwngenx = qwnx.generate(sources, biogens)

In [ ]:
test_df["biobart_gen"] = biogens
test_df["qwen3_gen"] = qwngens
test_df["qwen3x_gen"] = qwngenx
test_df.info()

In [ ]:
test_df.to_json("data/test-gen3-qwen-only.jsonl", orient="records", lines=True)